# Week 11 — Clustering-based Query Selection (with Plots)

**What this notebook does**
1. Loads all my Week 1–10 query history (inputs + outputs)
2. Finds natural groupings with clustering (KMeans + DBSCAN)
3. Visualises clusters (2D scatter or PCA projection)
4. Proposes Week 11 queries by targeting the best-performing cluster centroid and avoiding duplicates

**How proposals are chosen**
- Pick a clustering method (default: KMeans)
- Choose the cluster with the highest mean output
- Propose a new point near that cluster's centroid, but not too close to existing points


In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Loaded history
df = pd.read_csv('data/bbo_history_week1_10.csv')
df.head()


,week,function,y,d,x1,x2,x3,x4,x5,x6,x7,x8
0,1,1,3.100722e-34,2,0.759779,0.804108,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2,5.697925e-01,2,0.717915,0.002064,NaN,NaN,NaN,NaN,NaN,NaN
2,1,3,-1.518442e-01,3,0.994942,0.051967,0.792526,NaN,NaN,NaN,NaN,NaN
3,1,4,-7.158151e-01,4,0.404477,0.413254,0.303108,0.434359,NaN,NaN,NaN,NaN
4,1,5,3.653951e+03,4,0.940282,0.064909,0.998637,0.996528,NaN,NaN,NaN,NaN


## Helper functions

In [9]:

def get_Xy(df, function_id):
    sub = df[df["function"]==function_id].copy()
    d = int(sub["d"].iloc[0])
    X = sub[[f"x{i}" for i in range(1,d+1)]].to_numpy()
    y = sub["y"].to_numpy()
    return sub, X, y, d

def format_query(x):
    return "-".join(f"{float(v):.6f}" for v in x)

def choose_best_cluster(y, labels):
    best_c, best_mean = None, -np.inf
    for c in np.unique(labels):
        if c == -1:  # DBSCAN noise
            continue
        m = y[labels==c].mean()
        if m > best_mean:
            best_mean = m; best_c = c
    return best_c, best_mean

def propose_best_in_best_cluster(X, y, labels, step=0.01, n_candidates=50000, seed=42):
    rng = np.random.default_rng(seed)

    # 1) choose best cluster by mean y (ignore DBSCAN noise)
    clusters = [c for c in np.unique(labels) if c != -1]
    if len(clusters) == 0:
        best_idx = int(np.argmax(y))
        x_best = X[best_idx]
    else:
        best_cluster = max(clusters, key=lambda c: y[labels==c].mean())

        # 2) choose best point inside that cluster
        idxs = np.where(labels == best_cluster)[0]
        best_idx = idxs[int(np.argmax(y[idxs]))]
        x_best = X[best_idx]

    # 3) propose candidates around x_best (small ball) and avoid duplicates
    d = X.shape[1]
    dirs = rng.normal(size=(n_candidates, d))
    dirs = dirs / (np.linalg.norm(dirs, axis=1, keepdims=True) + 1e-12)
    cand = np.clip(x_best + step * dirs, 0.0, 1.0)

    # avoid duplicates: maximise distance to nearest existing point
    d2 = ((cand[:,None,:] - X[None,:,:])**2).sum(axis=2)
    min_d = np.sqrt(d2.min(axis=1))

    # mild preference to stay close (so we don't jump away)
    dist_best = np.linalg.norm(cand - x_best, axis=1)
    score = min_d - 0.05 * dist_best

    return cand[int(np.argmax(score))], x_best


## Visualise clusters (PCA used for d>2)

In [10]:

def plot_clusters(function_id, method='kmeans', k=2, eps=0.7, min_samples=3):
    sub, X, y, d = get_Xy(df, function_id)
    Xs = StandardScaler().fit_transform(X)

    if method == 'kmeans':
        labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(Xs)
        title = f"F{function_id} — KMeans(k={k})"
    else:
        labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(Xs)
        title = f"F{function_id} — DBSCAN(eps={eps}, min_samples={min_samples})"

    if d > 2:
        Z = PCA(n_components=2, random_state=42).fit_transform(Xs)
        xlab, ylab = "PC1", "PC2"
    else:
        Z = X
        xlab, ylab = "x1", "x2"

    plt.figure(figsize=(6,5))
    sc = plt.scatter(Z[:,0], Z[:,1], c=y, s=120, edgecolor='k')
    plt.colorbar(sc, label='Output y')
    for i, wk in enumerate(sub['week'].astype(int).to_list()):
        plt.text(Z[i,0], Z[i,1], str(wk), fontsize=8)

    plt.title(title + "  (labels show week)")
    plt.xlabel(xlab); plt.ylabel(ylab)
    plt.show()
    return labels


## Propose Week 11 queries from the best cluster

In [7]:

proposals = {}

for f in range(1,9):
    sub, X, y, d = get_Xy(df, f)
    Xs = StandardScaler().fit_transform(X)

    # K choice: small for low-d, slightly larger for high-d
    k = 2 if d <= 3 else 3
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(Xs)

    step = 0.02 if d >= 6 else 0.03
    x_next, x_best = propose_best_in_best_cluster(X, y, labels, step=0.01 if d>=6 else 0.02, seed=100+f)

    proposals[f] = {
        "d": d,
        "best_y": float(np.max(y)),
        "best_week": int(sub.iloc[int(np.argmax(y))]["week"]),
        "centroid": centroid.tolist(),
        "proposal": x_next.tolist(),
        "portal": format_query(x_next),
        "cluster": int(best_cluster) if best_cluster is not None else None
    }

for f in range(1,9):
    print(f"Function {f}: {proposals[f]['portal']}   (best_y={proposals[f]['best_y']:.6g})")


Function 1: 0.628811-0.610035   (best_y=1.97963)
Function 2: 0.697934-0.002930   (best_y=0.609311)
Function 3: 0.355682-0.581219-0.461686   (best_y=-0.000247902)
Function 4: 0.377657-0.396035-0.356354-0.446113   (best_y=0.217208)
Function 5: 0.999284-0.019993-1.000000-1.000000   (best_y=4440.11)
Function 6: 0.541364-0.167740-0.639112-0.885874-0.031735   (best_y=-0.475767)
Function 7: 0.014754-0.185006-0.289782-0.182615-0.395384-0.666262   (best_y=2.10642)
Function 8: 0.036317-0.441900-0.015780-0.330508-0.991602-0.041856-0.098044-0.704937   (best_y=9.59028)
